In [34]:
import pandas as pd
import numpy as np


In [36]:
interactions = pd.read_csv("../data/interactions.csv")
products = pd.read_csv("../data/products.csv")


In [38]:
rating_map = {
    "click": 1,
    "add_to_cart": 3,
    "purchase": 5
}

interactions["rating"] = interactions["event_type"].map(rating_map)


In [40]:
popular_products = (
    interactions[interactions["event_type"] == "purchase"]
    .groupby("product_id")
    .size()
    .sort_values(ascending=False)
)


In [42]:
def recommend_popular(top_n=10):
    return popular_products.head(top_n).index.tolist()


In [44]:
recommend_popular(5)


[296, 157, 238, 155, 16]

In [46]:
user_item_matrix = interactions.pivot_table(
    index="user_id",
    columns="product_id",
    values="rating",
    aggfunc="mean"
).fillna(0)


In [48]:
user_item_matrix.shape


(500, 300)

In [50]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)


In [52]:
def recommend_collaborative(user_id, top_n=10):
    if user_id not in user_item_matrix.index:
        return recommend_popular(top_n)
    
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]
    
    weighted_scores = user_item_matrix.loc[similar_users.index].T.dot(similar_users)
    
    weighted_scores = weighted_scores.sort_values(ascending=False)
    
    return weighted_scores.head(top_n).index.tolist()


In [54]:
recommend_collaborative(user_id=1, top_n=5)


[207, 71, 204, 23, 250]

In [56]:
product_margin = products.set_index("product_id")["profit_margin"]


In [58]:
def rerank_with_business(product_list, alpha=0.7):
    scores = []
    for pid in product_list:
        relevance = 1
        margin = product_margin.get(pid, 0)
        final_score = alpha * relevance + (1 - alpha) * margin
        scores.append((pid, final_score))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    return [pid for pid, _ in scores]


In [60]:
def recommend_final(user_id, top_n=10):
    if user_id not in interactions["user_id"].values:
        return recommend_popular(top_n)
    
    candidates = recommend_collaborative(user_id, top_n=20)
    final_recommendations = rerank_with_business(candidates)
    
    return final_recommendations[:top_n]


In [62]:
recommend_final(user_id=1, top_n=5)


[207, 71, 90, 79, 113]

In [66]:
interactions["timestamp"] = pd.to_datetime(interactions["timestamp"])
interactions = interactions.sort_values("timestamp")

split_time = interactions["timestamp"].quantile(0.8)

train_data = interactions[interactions["timestamp"] <= split_time]
test_data = interactions[interactions["timestamp"] > split_time]


In [68]:
test_purchases = test_data[test_data["event_type"] == "purchase"]

ground_truth = (
    test_purchases.groupby("user_id")["product_id"]
    .apply(list)
    .to_dict()
)


In [70]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    if not recommended_k:
        return 0
    return len(set(recommended_k) & set(relevant)) / k


In [78]:
def recall_at_k(recommended, relevant, k):
    if not relevant:
        return 0
    return len(set(recommended[:k]) & set(relevant)) / len(relevant)


In [80]:
import math

def ndcg_at_k(recommended, relevant, k):
    dcg = 0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            dcg += 1 / math.log2(i + 2)
    
    ideal_dcg = sum(1 / math.log2(i + 2) for i in range(min(len(relevant), k)))
    
    return dcg / ideal_dcg if ideal_dcg > 0 else 0


In [84]:
def evaluate_model(recommend_func, k=10):
    precisions, recalls, ndcgs = [], [], []
    
    for user_id, relevant_items in ground_truth.items():
        recommendations = recommend_func(user_id, k)
        
        precisions.append(precision_at_k(recommendations, relevant_items, k))
        recalls.append(recall_at_k(recommendations, relevant_items, k))
        ndcgs.append(ndcg_at_k(recommendations, relevant_items, k))
    
    return {
        "Precision@K": np.mean(precisions),
        "Recall@K": np.mean(recalls),
        "NDCG@K": np.mean(ndcgs)
    }


In [86]:
baseline_results = evaluate_model(
    lambda u, k: recommend_popular(k),
    k=10
)

baseline_results


{'Precision@K': 0.011931818181818182,
 'Recall@K': 0.06922348484848485,
 'NDCG@K': 0.039179678378561136}

In [88]:
final_results = evaluate_model(
    lambda u, k: recommend_final(u, k),
    k=10
)

final_results


{'Precision@K': 0.046306818181818185,
 'Recall@K': 0.27514204545454546,
 'NDCG@K': 0.15734741168329333}

In [90]:
def average_margin(recommend_func, k=10):
    margins = []
    
    for user_id in ground_truth.keys():
        recs = recommend_func(user_id, k)
        margins.extend(
            products[products["product_id"].isin(recs)]["profit_margin"]
        )
    
    return np.mean(margins)


In [92]:
baseline_margin = average_margin(lambda u, k: recommend_popular(k))
final_margin = average_margin(lambda u, k: recommend_final(u, k))


In [94]:
baseline_margin, final_margin


(0.179, 0.30087215909090914)